# Imports

In [90]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models

# For matrix operations
import numpy as np

# Data visualizaton.
import matplotlib.pyplot as plt
from matplotlib import style
import seaborn as sns
import random as rn

# Custom Image Dataset

In [91]:
import os
from torchvision.io import read_image
import pandas as pd
from PIL import Image
import math


def get_default_device():
  """Pick GPU if available, else CPU"""
  if torch.cuda.is_available():
      return torch.device('cuda')
  else:
      return torch.device("mps")

device = get_default_device()
print(device)

class ImageDataset(Dataset):
    def __init__(self, index_df, root=None, tranformations = None,
                 augmentation = False, default_transformation = None):
        # super().__init__()
        self.root = root
        self.augmentation = augmentation
        self.device = get_default_device()
        self.transformation = None
        self.default_transformation = default_transformation
        if root is None:
          self.root = os.path.join( 'data','train','train')
        #initializing the transformation set with the default conversion
        if augmentation:
          self.transformation = [
              default_transformation
          ]
          if tranformations:
            self.transformation.extend(tranformations)

        self.index_file = index_df

    def __len__(self):
        return self.index_file.shape[0]  # Always return the original number of images

    def __getitem__(self, index):
        img_path = os.path.join(self.root, str(self.index_file.iloc[index, 0]), self.index_file.iloc[index, 1])
        label = self.index_file.iloc[index, 0]
        image = Image.open(img_path).convert('RGB')

        # Apply only one augmentation (randomly selected)
        if self.augmentation and self.transformation:
            transform = self.transformation[rn.randint(0, len(self.transformation) - 1)]  # Randomly select one transform
            image = transform(image)
        elif self.default_transformation:
            image = self.default_transformation(image)

        return image, label

cuda


# Image Augmentation

In [108]:
mean = (0.5, 0.5, 0.5)
std = (0.5, 0.5, 0.5)
final_size = (224,224)
image_augmentation = [
  transforms.Compose(
            [
                transforms.Resize(final_size),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(mean=mean, std=std),
            ]
        ),
  transforms.Compose(
            [
                transforms.Resize(final_size),
                transforms.ColorJitter(
                    brightness=0.5, contrast=0.7, saturation=0.7, hue=0.5
                ),
                transforms.ToTensor(),
                transforms.Normalize(mean=mean, std=std),
            ]
        ),
  transforms.Compose(
            [
                transforms.Resize(final_size),
                transforms.RandomRotation(degrees=30),
                transforms.ToTensor(),
                transforms.Normalize(mean=mean, std=std),
            ]
        ),
  transforms.Compose(
            [
                transforms.Resize(final_size),
                transforms.RandomGrayscale(p=0.7),
                transforms.ToTensor(),
                transforms.Normalize(mean=mean, std=std),
            ]
        ),
  transforms.Compose(
            [
                transforms.Resize(final_size),
                transforms.ToTensor(),
                transforms.RandomErasing(
                    p=0.5, scale=(0.02, 0.33), ratio=(0.3, 3.3), value="random"
                ),
                transforms.Normalize(mean=mean, std=std),
            ]
        ),
  transforms.Compose(
            [
                transforms.Resize(final_size),
                transforms.ToTensor(),
                transforms.GaussianBlur(kernel_size=7),
                transforms.Normalize(mean=mean, std=std),
            ]
        ),
   transforms.Compose(
            [
                transforms.Resize(final_size),
                transforms.ToTensor(),
                transforms.RandomInvert(p=0.7),
                transforms.Normalize(mean=mean, std=std),
            ]
        ),
        transforms.Compose(
            [
                transforms.RandomResizedCrop(size=final_size),
                transforms.ToTensor(),
                transforms.Normalize(mean=mean, std=std),
            ]
        ),
        transforms.Compose(
            [
                transforms.CenterCrop(size=(100, 100)),
                transforms.Resize(final_size),
                transforms.ToTensor(),
                transforms.Normalize(mean=mean, std=std),
            ]
        ),
        transforms.Compose(
            [
                transforms.RandomPerspective(
                    distortion_scale=0.5,
                    p=0.5,
                    interpolation=transforms.InterpolationMode.BILINEAR,
                ),
                transforms.Resize(final_size),
                transforms.ToTensor(),
                transforms.Normalize(mean=mean, std=std),
            ]
        ),
  transforms.Compose(
            [
                transforms.Resize(final_size),
                transforms.ToTensor(),
                transforms.Normalize(mean=mean, std=std)
            ]
        ),
  transforms.Compose(
        [
            transforms.Resize(final_size),
            transforms.ColorJitter(
                brightness=0.5, contrast=0.7, saturation=0.7, hue=0.5
            ),
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std),
        ]
    )

]

def get_train_test_split(train_size = 0.8):
  labels = sorted(list(range(100)) * 10)
  file_names = [str(f) + '.jpg' for f in list(range(10))*100]
  data = pd.DataFrame(zip(labels,file_names), columns=['Target','File'])
  all_samples = [i for i in range(data.shape[0])]
  rn.shuffle(all_samples)
  train_offset = int(train_size * len(all_samples))
  train_samples = all_samples[:train_offset]
  test_samples = all_samples[train_offset:]

  train_samples = data.iloc[train_samples,:]
  test_samples = data.iloc[test_samples,:]

  train_samples.reset_index(drop=True, inplace=True)
  test_samples.reset_index(drop=True, inplace=True)
  return train_samples, test_samples


default_transformation = transforms.Compose(
    [
        transforms.Resize(final_size),
        transforms.ToTensor(),
        #transforms.Normalize(mean=mean, std=std)
    ]
)

train_samples, test_samples = get_train_test_split()
train_data = ImageDataset(train_samples, tranformations = image_augmentation,
                          augmentation=True, default_transformation=default_transformation)
test_data = ImageDataset(test_samples, default_transformation=default_transformation)

In [109]:
print(len(train_data))
for i in range(0,len(train_data),800):
  img, label = train_data.__getitem__(i)
  print(f'Img={img.shape}; Label={label}')
  print('-'*25)

800
Img=torch.Size([3, 224, 224]); Label=42
-------------------------


# Training Loop

In [110]:
from tqdm import tqdm
import gc

def cleanup(model):
  del model
  gc.collect()
  torch.cuda.empty_cache()



def training_loop(model, train_data, test_data, b_size=32,
                  num_epochs=20, model_name='model1', lr=1e-4):
    
    # Assuming `device` is already defined
    train_loader = DataLoader(train_data, batch_size=b_size, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=b_size, shuffle=False)
    
    optimizer = optim.Adam(model.parameters(),lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    # Reduce LR when validation accuracy stops improving
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',  # Maximize validation accuracy
        factor=0.5,  # Halve the learning rate when triggered
        patience=3,  # Wait for 3 epochs with no improvement
        verbose=True
    )

    # Function to calculate accuracy
    def calculate_accuracy(loader, model, pbar=None):
        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for data in loader:
                images, labels = data
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels.squeeze()).sum().item()
                if pbar:
                    pbar.update(1)
        return 100 * correct / total

    # Fine-tuning the network
    train_losses, test_losses = [], []
    train_acc, test_acc = [], []
    best_acc = 0
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        print('Fine-tuning Epoch #', epoch)
        with tqdm(total=math.ceil(len(train_data)/b_size)) as pbar:
            for i, data in enumerate(train_loader, 0):
                inputs, labels = data
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels.squeeze())
                loss.backward()
                optimizer.step()
                running_loss += loss.item()
                pbar.update(1)

        train_loss = running_loss / len(train_loader)
        train_losses.append(train_loss)
        print('Computing Train Accuracy...')
        with tqdm(total=math.ceil(len(train_data)/b_size)) as pbar:
            train_accuracy = calculate_accuracy(train_loader, model, pbar)
        train_acc.append(train_accuracy)

        test_loss = 0.0
        print('Testing Epoch #', epoch)
        model.eval()
        print('Computing Test Loss...')
        with torch.no_grad():
            with tqdm(total=math.ceil(len(test_data)/b_size)) as pbar:
                for data in test_loader:
                    images, labels = data
                    images, labels = images.to(device), labels.to(device)
                    outputs = model(images)
                    loss = criterion(outputs, labels.squeeze())
                    test_loss += loss.item()
                    pbar.update(1)

        test_loss /= len(test_loader)
        test_losses.append(test_loss)
        print('Computing Test Accuracy...')
        with tqdm(total=math.ceil(len(test_data)/b_size)) as pbar:
            test_accuracy = calculate_accuracy(test_loader, model, pbar)
        test_acc.append(test_accuracy)

        print(f'\nEpoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.2f}%, Test Loss: {test_loss:.4f}, Test Acc: {test_accuracy:.2f}% \n')
        print('_'*30)
        print()

        # Save the best model based on test accuracy
        if test_accuracy > best_acc:
            torch.save(model.state_dict(), f'{model_name}-{int(test_accuracy)}.pth')
            best_acc = test_accuracy
            print('*'*30)
            print(f'\nBest Epoch so far: {epoch + 1}, Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.2f}%, Test Loss: {test_loss:.4f}, Test Acc: {test_accuracy:.2f}% \n')
            print('*'*30)
            print('\n')

    return train_acc, train_losses, test_acc, test_losses

def fine_tune_model(model, train_data, test_data, b_size = 32,
                    num_epochs = 20, model_name = 'model1', lr = 1e-5):
    # Fine-tune the model on the new dataset

    train_loader = DataLoader(train_data, batch_size = b_size, shuffle=True)
    test_loader = DataLoader(test_data, batch_size = b_size, shuffle = False)

    # Use Adam optimizer with a low learning rate
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # Function to calculate accuracy
    def calculate_accuracy(loader, model, pbar=None):
        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for data in loader:
                images, labels = data
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels.squeeze()).sum().item()
                if pbar:
                    pbar.update(1)
        return 100 * correct / total

    # Fine-tuning the network
    train_losses, test_losses = [], []
    train_acc, test_acc = [], []
    best_acc = 0
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        print('Epoch #', epoch)
        with tqdm(total=math.ceil(len(train_data)/b_size)) as pbar:
            for i, data in enumerate(train_loader, 0):
                inputs, labels = data
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels.squeeze())
                loss.backward()
                optimizer.step()
                running_loss += loss.item()
                pbar.update(1)

        train_loss = running_loss / len(train_loader)
        train_losses.append(train_loss)
        print('Computing Train Accuracy...')
        with tqdm(total=math.ceil(len(train_data)/b_size)) as pbar:
            train_accuracy = calculate_accuracy(train_loader, model, pbar)
        train_acc.append(train_accuracy)

        test_loss = 0.0
        print('Testing Epoch #', epoch)
        model.eval()
        print('Computing Test Loss...')
        with torch.no_grad():
            with tqdm(total=math.ceil(len(test_data)/b_size)) as pbar:
                for data in test_loader:
                    images, labels = data
                    images, labels = images.to(device), labels.to(device)
                    outputs = model(images)
                    loss = criterion(outputs, labels.squeeze())
                    test_loss += loss.item()
                    pbar.update(1)

        test_loss /= len(test_loader)
        test_losses.append(test_loss)
        print('Computing Test Accuracy...')
        with tqdm(total=math.ceil(len(test_data)/b_size)) as pbar:
            test_accuracy = calculate_accuracy(test_loader, model, pbar)
        test_acc.append(test_accuracy)

        print(f'\nEpoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.2f}%, Test Loss: {test_loss:.4f}, Test Acc: {test_accuracy:.2f}% \n')
        print('_'*30)
        print()

        # Save the best model based on test accuracy
        if test_accuracy > best_acc:
            torch.save(model.state_dict(), f'{model_name}-{int(test_accuracy)}.pth')
            best_acc = test_accuracy
            print('*'*30)
            print(f'\nBest Epoch so far: {epoch + 1}, Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.2f}%, Test Loss: {test_loss:.4f}, Test Acc: {test_accuracy:.2f}% \n')
            print('*'*30)
            print('\n')

    return train_acc, train_losses, test_acc, test_losses




In [112]:
class ModelHead(nn.Module):
    def __init__(self, in_features, num_classes=100):
        super(ModelHead, self).__init__()
        self.custom_classifier = nn.Sequential(
          nn.Linear(in_features, 1024),
          nn.ReLU(inplace=True),
          nn.Dropout(0.2),
          nn.Linear(1024, num_classes),
        )

    def forward(self, x):
        return self.custom_classifier(x)

# ResNet 50

In [113]:
# Load ResNet-50
model = models.resnet50(weights='IMAGENET1K_V2')

# Freeze all layers except the final fully connected layer
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the last three layers (layer4 and last two conv layers in layer3)
for name, param in model.named_parameters():
    if 'layer4' in name or 'layer3.5' in name or 'layer3.4' in name:
        param.requires_grad = True

# Replace the final layer with a custom head
num_features = model.fc.in_features
model.fc = ModelHead(num_features)
model.to(device)

# Use mixed precision training
# scaler = GradScaler()

# Train the model with a smaller batch size and fewer epochs
train_acc, train_losses, test_acc, test_losses = training_loop(
    model, train_data, test_data, num_epochs=20, b_size=16, model_name='resnet50'
)

# Now, fine-tune the model by unfreezing all layers and training with a low learning rate
# Unfreeze the entire model for fine-tuning
#for param in model.parameters():
#    param.requires_grad = True

# Fine-tune the model with a very small learning rate and a few epochs
#fine_tune_acc, fine_tune_losses, fine_tune_test_acc, fine_tune_test_losses = fine_tune_model(
#    model, train_data, test_data, num_epochs=1, b_size=16, model_name='resnet50', lr=1e-5
#)

Fine-tuning Epoch # 0


100%|██████████| 50/50 [00:11<00:00,  4.49it/s]


Computing Train Accuracy...


100%|██████████| 50/50 [00:10<00:00,  4.89it/s]


Testing Epoch # 0
Computing Test Loss...


...

Computing Test Accuracy...


100%|██████████| 13/13 [00:02<00:00,  6.08it/s]



Epoch 2/20, Train Loss: 4.5764, Train Acc: 5.12%, Test Loss: 4.5789, Test Acc: 3.50% 

______________________________

******************************

Best Epoch so far: 2, Train Loss: 4.5764, Train Acc: 5.12%, Test Loss: 4.5789, Test Acc: 3.50% 

******************************


Fine-tuning Epoch # 2


 22%|██▏       | 11/50 [00:02<00:09,  4.07it/s]


KeyboardInterrupt: 

In [111]:
cleanup(model)

In [106]:
def plot_losses(train_losses, test_losses):
    plt.figure(figsize=(8, 6))
    plt.plot(train_losses, label='Train Accuracy', marker='o', linestyle='-')
    plt.plot(test_losses, label='Validation Accuracy', marker='o', linestyle='-')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(' Train vs Val Accuracy (Resnet50) ')
    plt.legend()
    plt.grid(True)
    plt.show()
    
plot_losses(train_acc, test_acc)

NameError: name 'train_acc' is not defined

# Predictions

Before this section can be executed, one needs to run the training loop to save the tuned models to disk. The filename will be needed to load the model and do the predictions.

In [11]:
class PredictionDataset(Dataset):
  def __init__(self, root, default_transformation = None):
    self.root = root
    self.default_transformation = default_transformation

  def __len__(self):
    return 1000

  def __getitem__(self, index):
    img_path = os.path.join(self.root, str(index) + '.jpg')
    label = f'{str(index)}.jpg'
    image = Image.open(img_path).convert('RGB')
    if self.default_transformation:
      image = self.default_transformation(image)
    return image, label


def generate_prediction(model, b_size = 16):
  model.eval()
  default_transformation = transforms.Compose(
      [
          transforms.Resize(final_size),
          transforms.ToTensor(),
          #transforms.Normalize(mean=mean, std=std)
      ]
  )
  pred_data = PredictionDataset("data/test/test", default_transformation)
  pred_loader = DataLoader(pred_data, batch_size = b_size, shuffle = False)
  pred_names = []
  pred_labels = []
  with torch.no_grad():
    for data in pred_loader:
      images, names = data
      #print(type(names))
      #print(type(images))
      images = images.to(device)
      outputs = model(images)
      _, predicted = torch.max(outputs.data, 1)
      #print(type(predicted))
      #print(list(names))
      #print(list(predicted))
      pred_names.extend(names)
      pred_labels.extend(predicted.cpu().numpy())
  return pd.DataFrame({'ID':pred_names, 'Label':pred_labels})

## Resnet Prediction

In [13]:
model_name = 'resnet50-66.pth'
pred_file = 'submission.csv'

model = torchvision.models.resnet50()
num_features = model.fc.in_features
model.fc = ModelHead(num_features)
print(model.load_state_dict(torch.load(model_name)))
model.to(device)

predictions = generate_prediction(model)
predictions.to_csv(pred_file,index=False)

<All keys matched successfully>
